# ISIC 2024 Skin Cancer - Kaggle Runner

Notebook này dùng để chạy tuần tự các bước của project trên Kaggle.

## Hướng dẫn setup:
1. Upload toàn bộ folder `Skin-cancer-classification-isic2024` của bạn lên Kaggle dưới dạng **Dataset**. (Ví dụ đặt tên là `skin-cancer-code`)
2. Thêm dataset chứa code đó vào notebook này. Cập nhật biến `KAGGLE_CODE_DATASET` ở Cell 1 thành đúng đường dẫn dataset của bạn.
3. Thêm dataset cuộc thi **ISIC 2024 - Skin Cancer Detection** vào notebook.
4. Bật GPU (T4 x2 hoặc P100) và nhấn **Run All**.

In [ ]:
import os
import shutil
from pathlib import Path
import glob

# =========================
# CẤU HÌNH ĐƯỜNG DẪN KAGGLE
# =========================

# SỬA DÒNG NÀY THÀNH ĐƯỜNG DẪN DATASET CODE CỦA BẠN TRÊN KAGGLE
KAGGLE_CODE_DATASET = "/kaggle/input/skin-cancer-code"

# NẾU BẠN CHẠY KAGGLE KHÔNG CÓ INTERNET, TRỎ ĐƯỜNG DẪN VÀO FILE TRỌNG SỐ OFFLINE
# Ví dụ: KAGGLE_BACKBONE_WEIGHTS = "/kaggle/input/backbone-weights/efficientnet_b0_backbone.pth"
KAGGLE_BACKBONE_WEIGHTS = None

KAGGLE_ISIC_DATASET = "/kaggle/input/isic-2024-challenge"
WORKING_CODE_DIR = "/kaggle/working/code"

if Path("/kaggle/input").exists():
    print("✅ Môi trường Kaggle được phát hiện.")
    
    # 1. Copy source code vào working directory để có quyền write (do cần sửa config)
    if not Path(WORKING_CODE_DIR).exists():
        print(f"Copying code from {KAGGLE_CODE_DATASET} to {WORKING_CODE_DIR}...")
        try:
            shutil.copytree(KAGGLE_CODE_DATASET, WORKING_CODE_DIR)
            print("✅ Đã copy source code.")
        except Exception as e:
            print(f"❌ Lỗi khi copy code: {e}")
            print("Hãy kiểm tra lại đường dẫn KAGGLE_CODE_DATASET.")
            
    # 2. Setup thư mục outputs và symlink dataset ISIC
    os.environ["ISIC_DATA_ROOT"] = "/kaggle/working/data"
    os.environ["ISIC_CHECKPOINT_DIR"] = "/kaggle/working/checkpoints"
    os.environ["ISIC_ARTIFACT_DIR"] = "/kaggle/working/artifacts"
    os.environ["ISIC_OUTPUT_DIR"] = "/kaggle/working/outputs"
    
    for d in ["/kaggle/working/data/raw", "/kaggle/working/data/processed", 
              "/kaggle/working/data/splits", "/kaggle/working/checkpoints", 
              "/kaggle/working/artifacts", "/kaggle/working/outputs"]:
        os.makedirs(d, exist_ok=True)
        
    if Path(KAGGLE_ISIC_DATASET).exists():
        for f in glob.glob(f"{KAGGLE_ISIC_DATASET}/*"):
            basename = os.path.basename(f)
            target = f"/kaggle/working/data/raw/{basename}"
            if not os.path.exists(target):
                os.symlink(f, target)
        
        # Đảm bảo folder ảnh được map đúng tên mong đợi của config.py
        kaggle_train_img = f"{KAGGLE_ISIC_DATASET}/train-image/image"
        target_img = "/kaggle/working/data/raw/ISIC_2024_Training_Input"
        if os.path.exists(kaggle_train_img) and not os.path.exists(target_img):
            os.symlink(kaggle_train_img, target_img)
            
        print("✅ Đã link dataset ISIC 2024.")
    else:
        print(f"❌ Không tìm thấy dataset ISIC tại {KAGGLE_ISIC_DATASET}.")
else:
    print("✅ Đang chạy Local.")
    WORKING_CODE_DIR = "."

os.chdir(WORKING_CODE_DIR)
print(f"Thư mục hiện tại: {os.getcwd()}")

In [ ]:
# =========================
# OVERRIDE CONFIG CHO GPU KAGGLE
# =========================
if Path("/kaggle/input").exists() and Path(WORKING_CODE_DIR).exists():
    # config.py nay da duoc move ra thu muc root
    config_path = Path("config.py")
    
    if config_path.exists():
        with open(config_path, "r", encoding="utf-8") as f:
            content = f.read()
            
        # Tự động thay đổi các param dành cho local CPU sang Kaggle GPU
        replacements = {
            "IMAGE_SIZE = 160": "IMAGE_SIZE = 224",
            "BATCH_SIZE = 4": "BATCH_SIZE = 32",
            "EPOCHS = 2": "EPOCHS = 10",
            "NEGATIVE_SAMPLE_SIZE = 500": "NEGATIVE_SAMPLE_SIZE = 10000",
            "EVAL_NEGATIVE_SAMPLE_SIZE = 200": "EVAL_NEGATIVE_SAMPLE_SIZE = 3000",
            "NUM_WORKERS = 0": "NUM_WORKERS = 2",
            "MAX_TRAIN_BATCHES = 20": "MAX_TRAIN_BATCHES = None",
            "MAX_VAL_BATCHES = 10": "MAX_VAL_BATCHES = None",
            "USE_PRETRAINED = False": "USE_PRETRAINED = True",
            'BACKBONE = "mobilenet_v3_small"': 'BACKBONE = "efficientnet_b0"',
            "UNFREEZE_EPOCH = 999": "UNFREEZE_EPOCH = 3"
        }
        
        for old, new in replacements.items():
            content = content.replace(old, new)
            
        # Ghi đè cấu hình offline backbone weights nếu có
        if KAGGLE_BACKBONE_WEIGHTS:
            content = content.replace("BACKBONE_WEIGHTS_PATH = None", f'BACKBONE_WEIGHTS_PATH = r"{KAGGLE_BACKBONE_WEIGHTS}"')
            # Nếu đã có weights offline thì không cần load pretrained qua mạng nữa
            content = content.replace("USE_PRETRAINED = True", "USE_PRETRAINED = False")
            
        with open(config_path, "w", encoding="utf-8") as f:
            f.write(content)
            
        print("✅ Đã cấu hình config.py cho GPU Training.")
        if KAGGLE_BACKBONE_WEIGHTS:
            print(f"📦 Kích hoạt Offline Backbone Weights: {KAGGLE_BACKBONE_WEIGHTS}")
    else:
        print(f"❌ Không tìm thấy {config_path}")

---
## Bước 1: Kiểm tra tính toàn vẹn dữ liệu (Data Integrity)

In [ ]:
!python 01_data_integrity.py

---
## Bước 2: Chia dữ liệu theo Patient-level (Split Data)

In [ ]:
!python 02_split_data.py

---
## Bước 3: Tiền xử lý Metadata (Preprocess Metadata)

In [ ]:
!python 03_preprocess_metadata.py

---
## Bước 4: Kiểm tra Dataloader đa phương thức

In [ ]:
!python 04_test_dataloader.py

---
## Bước 5: Kiểm tra Model Architecture

In [ ]:
!python 05_test_model.py

---
## Bước 6: Training (GPU)

In [ ]:
!python 06_train_demo.py

---
## Bước 7: Đánh giá Holdout

In [ ]:
!python 07_evaluate_holdout.py

---
## Bước 8: Dự đoán & Tạo Submission

In [ ]:
!python 08_infer_submission.py

---
## Xem kết quả Submission

In [ ]:
import pandas as pd
import shutil

sub_path = "/kaggle/working/outputs/submission.csv"
if os.path.exists(sub_path):
    print("✅ Đã tạo thành công submission.csv")
    # Copy ra ngoài thư mục root để Kaggle nhận diện nhanh
    shutil.copy(sub_path, "/kaggle/working/submission.csv")
    
    df = pd.read_csv("/kaggle/working/submission.csv")
    display(df.head())
else:
    print("❌ Không tìm thấy file submission.csv")